# 02. 연속소둔로 정비 → 부품 연결

**목적**
- 연속소둔로(`P1-CR1-CAL01`)의 정비 기록이 어떤 부품 교체로 이어졌는지 순서대로 따라가보자
- BM / TBM / CBM 전부 확인할거임. 교체 없는 정비도 남김.
- 데이터는 KAMP 공개 제조AI데이터셋을 재구성한 교육용 재현 데이터임.
- 여기서는 정비·부품의 기록과 의미를 확인함. 센서 이상 라벨을 정비만으로 확정하지 않음.

**따라가는 순서**
연속소둔로 정비이력 → WO_NO → 부품교체이력 → PART_CD → 부품마스터

1. 데이터 불러오기
2. 연속소둔로 정비이력과 대기시간 박스플롯
3. 교체 없는 정비까지 남겨서 부품교체이력 연결
4. 부품마스터와 교체 간격 확인
5. 다음 분석에 넘길 정비 표 저장

설비·LOT·강종 연결 기준은 [01](01_data_alignment.ipynb)에서 확인했음.
LOT별 센서 움직임은 [03](03_lot_comparison.ipynb), 같은 강종 비교는 [04](04_grade_comparison.ipynb)에서 볼거임.
기존 종합 EDA 뒤쪽의 센서 분석을 여기서 반복하지 않음.

각 그래프 앞에는 **왜 이 그래프가 필요한가**, 뒤에는 **읽는 법**을 적었음.

In [ ]:
# 경로가 바뀌어도 저장소 루트와 개인 그림 폴더를 찾게 해둠.
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/README.md').is_file() and (p / 'notebooks').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from namelesspark_pipeline import load_sources, build_maintenance_context, build_lot_context, apply_user_exclusions

FIG_DIR = ROOT / 'figures/namelesspark/02_maintenance_parts'
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = ROOT / 'data/processed/analysis/maintenance/v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
    if cand in installed:
        rcParams['font.family'] = cand
        break
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110

In [ ]:
# 이 노트북에서만 쓰는 추가 설정
import matplotlib.dates as mdates
from IPython.display import display

CAL_TAG = "P1-CR1-CAL01"  # 연속소둔로 설비 태그

# 정비유형별 색 - 모든 그래프에서 같은 색 = 같은 정비유형
MNT_ORDER = ["TBM", "BM", "CBM"]
MNT_COLORS = {"TBM": "#2a78d6", "BM": "#eb6834", "CBM": "#1baf7a"}

# 그래프 기본 모양: 위·오른쪽 테두리 없애고 옅은 격자
rcParams["axes.spines.top"] = False
rcParams["axes.spines.right"] = False
rcParams["axes.grid"] = True
rcParams["grid.color"] = "#e1e0d9"
rcParams["grid.linewidth"] = 0.6

# 그림은 이 노트북 전용 폴더에 저장함.

## 1. 데이터 불러오기

날짜 컬럼은 날짜형으로 읽어야 시간 계산(빼기, 정렬)을 할 수 있다고 함.
읽기와 정비·부품 연결 코드는 공통 함수에서 한 번만 관리함.

In [ ]:
sources = load_sources(ROOT)
eqp, part, event, repair, maint, partchg, temp = (
    sources[name] for name in ['eqp', 'part', 'event', 'repair', 'maint', 'partchg', 'temp']
)
maintenance_events, part_details = build_maintenance_context(maint, partchg, part)
print('연속소둔로 정비:', len(maintenance_events), '건 / 교체 부품 상세:', len(part_details), '행')

In [ ]:
# CR1 정기수리 일정 - 이후 그래프에서 회색으로 칠할 기간
cr1_repair = repair[repair["PLANT_CD"] == "CR1"]
display(cr1_repair)

## 2. Step 1 · 연속소둔로 정비이력

정비이력(maint 데이터프레임)에서 `EQP_TAG == "P1-CR1-CAL01"`만 남기고, 시간 차이 두 개를 새 컬럼으로 만들어보자

| 새 컬럼 | 계산 | 의미 |
| --- | --- | --- |
| `대기시간` | `STRT_DT - DETC_DT` | 탐지(접수) 시각부터 작업을 시작하기까지 걸린 시간.  TBM(계획정비)은 이상을 탐지한 것이 아니라 작업이 접수된 시각에 가깝다고 볼 수 있음 |
| `작업시간` | `END_DT - STRT_DT` | 작업을 시작해서 끝내기까지 걸린 시간 |

> 데이터 사전: `DETC_DT` = 이상감지시각이고 "계획정비에는 값이 없다"고 돼 있는데, 실제로는 TBM에도 값이 있음 → 확인 필요

In [ ]:
cal = maintenance_events.copy()  # 부품이 없어도 남긴 정비 1건당 1행
cal = cal.sort_values("DETC_DT").reset_index(drop=True)

# 차이값을 시간 단위 숫자로 바꾸기
cal["대기시간"] = (cal["STRT_DT"] - cal["DETC_DT"]).dt.total_seconds() / 3600
cal["작업시간"] = (cal["END_DT"] - cal["STRT_DT"]).dt.total_seconds() / 3600



# FAIL_MODE 빈칸은 '고장이 없는 계획정비'일 수 있어서 지우지 않고 (공란)으로 채움
cal["FAIL_MODE"] = cal["FAIL_MODE"].fillna("(공란)")


# REMARK 앞뒤 공백 정리 ("  정기수리 부속작업" 처럼 앞에 공백이 붙은 값이 있음)
cal["REMARK"] = cal["REMARK"].str.strip()

print("전체 정비:", len(maint), "건 → 연속소둔로:", len(cal), "건")
display(cal.head())

### 2-1. 44건은 어떤 정비인가?

**왜 이 그래프가 필요한가?**
- 정비 44건을 한꺼번에 "정비"로 보면 안 됨
- 계획해서 한 정비(TBM), 고장 나서 한 정비(BM), 센서 징후를 보고 한 정비(CBM)는 의미가 완전히 다르다
- 우리 프로젝트가 만들려는 것은 **CBM 경보**이므로
    1. 각 유형이 몇 건인지
    2. 고장모드 / 부품교체가 얼마나 있는지
    - 크기를 알아야 뒤의 분석을 어떤 그룹으로 나눌지 정할 수 있음

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (1) 정비유형
counts = cal["MNT_TYPE"].value_counts().reindex(MNT_ORDER)
bars = axes[0].bar(counts.index, counts.values, color=[MNT_COLORS[t] for t in MNT_ORDER])
axes[0].bar_label(bars)  # 막대 위에 건수 표시
axes[0].set_title("정비유형 (MNT_TYPE)")

# (2) 고장모드
counts = cal["FAIL_MODE"].value_counts()
bars = axes[1].bar(counts.index, counts.values, color="#8c8c8c")
axes[1].bar_label(bars)
axes[1].set_title("고장모드 (FAIL_MODE)")

# (3) 부품교체 여부
counts = cal["PART_CHG"].value_counts()
bars = axes[2].bar(counts.index, counts.values, color="#8c8c8c")
axes[2].bar_label(bars)
axes[2].set_title("부품교체 여부 (PART_CHG)")

for ax in axes:
    ax.set_ylabel("건수")
fig.suptitle("연속소둔로 정비 44건 구성", y=1.04)
plt.savefig(os.path.join(FIG_DIR, "02_mnt_counts.png"), bbox_inches="tight")
plt.show()

**왜 교차표도 보는가?**
- "고장모드가 비어 있는 정비는 어떤 정비유형인가?"처럼 
- **두 항목의 관계**는 교차표로 봐야함

In [ ]:
# 정비유형 × 고장모드 교차표 (칸 안의 숫자 = 건수)
cross = pd.crosstab(cal["MNT_TYPE"], cal["FAIL_MODE"]).reindex(MNT_ORDER)
display(cross)

In [ ]:
# 유형별 교체 유무도 확인함. BM만 따로 뽑지 않음.
display(pd.crosstab(cal['MNT_TYPE'], cal['PART_CHG']).reindex(MNT_ORDER))
assert (cal['PART_CHG'].eq('Y') == cal['HAS_PART_RECORD']).all()

**읽는 법:**
- TBM 22건, BM 12건, CBM 10건, 부품교체는 8건
-`FAIL_MODE` 공란 24건 중 20건이 TBM(계획정비)
- 고장이 나서 한 정비가 아니므로 고장모드가 **구조적으로 비어 있는 것** (데이터 사전에도 "FAIL_MODE 결측은 구조적"이라고 나옴)
- 결측이라고 지우면 TBM이 거의 다 사라지므로 `(공란)`으로 추가하자
- 단, BM / CBM의 공란 4건은 REMARK가 "원인 미상", "확인 중"처럼 **원인이 아직 정해지지 않은 것**이라 성격이 다르다는 거 인지하셈
- `FAIL_MODE` 코드 (데이터 사전): `ELE` 전기 / `WER` 마모 / `MAL` 정렬불량 / `ETC` 기타
    - 단, 고장모드 배정 자체가 설정값(`[가정]`)이라 실제 고장 통계는 아님

### 2-2. 탐지 → 착수 대기시간은 왜 천차만별인가?

**왜 이 그래프가 필요한가?**
- CSV를 눈으로 보면 이상탐지(`DETC_DT`)와 작업착수(`STRT_DT`) 사이가 몇 분인 경우도, 며칠인 경우도 있다
- 이 차이가 **무엇 때문인지**(`MNT_TYPE`? `FAIL_MODE`? `PART_CHG`?) 알아야, "센서 징후가 정비보다 얼마나 먼저 잡히는가"를 제대로 해석할 수 있다
- 세 기준을 **같은 y축에 나란히** 놓으면 어느 기준이 대기시간을 가장 잘 가르는지 한눈에 비교 가능

그래프 보는 법
- **점 하나 = 정비 1건**
- claude 피셜: 점 색은 세 그래프 모두 정비유형
    - 각 정비이력의 대기시간 차이가 크다
    - 그래서 y축은 로그 눈금(1 → 10 → 100시간)을 사용했다고 함

In [ ]:
np.random.seed(0)  # 점을 흩뿌리는 위치가 실행할 때마다 같도록 고정

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)


for ax, col in zip(axes, ["MNT_TYPE", "FAIL_MODE", "PART_CHG"]):
    if col == "MNT_TYPE":
        groups = MNT_ORDER
    else:
        groups = sorted(cal[col].unique())

    # 상자그림: 가운데 선이 중앙값
    data = [cal.loc[cal[col] == g, "대기시간"] for g in groups]
    ax.boxplot(data, tick_labels=groups, showfliers=False, widths=0.5)

    # 점: 정비 1건, 색 = 정비유형 (겹치지 않게 옆으로 살짝 흩뿌림)
    for i, g in enumerate(groups, start=1):
        sub = cal[cal[col] == g]
        x = i + np.random.uniform(-0.15, 0.15, len(sub))
        ax.scatter(x, sub["대기시간"], c=sub["MNT_TYPE"].map(MNT_COLORS), s=35, edgecolor="white", zorder=3)

    ax.set_yscale("log")
    ax.set_yticks([0.01, 0.1, 1, 10, 100], ["0.01", "0.1", "1", "10", "100"])  # 눈금을 보통 숫자로
    ax.set_title(f"{col}별 대기시간")



axes[0].set_ylabel("이상탐지 → 작업착수 (시간, 로그 눈금)")


for t in MNT_ORDER:  # 범례만 만들기 위해 데이터 없는 빈 점을 찍음
    axes[2].scatter([], [], c=MNT_COLORS[t], label=t)
axes[2].legend(title="정비유형", loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "02_wait_hours_by_group.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 숫자로도 확인: 건수 / 중앙값 / 최댓값 (단위: 시간)
display(cal.groupby("MNT_TYPE")["대기시간"].agg(["count", "median", "max"]).reindex(MNT_ORDER).round(2))
display(cal.groupby("PART_CHG")["대기시간"].agg(["count", "median", "max"]).round(2))

**읽는 법**
- 대기시간을 가장 잘 가르는 기준은 **`MNT_TYPE`**
- TBM과 BM은 모두 수 시간 안에 착수하지만
### **CBM은 중앙값이 **약 117시간(약 5일)**, 최대 약 216시간**

- `FAIL_MODE`와 `PART_CHG` 그래프에서 위쪽에 떠 있는 점들은 **CBM(초록색)** 임
    - 이 두 기준에서 보이는 차이도 대부분 `CBM`이 어느 그룹에 섞였느냐에 따라 생김. 오히려 부품교체 Y가 N보다 짧다.
- 사실만 정리하면 **BM·TBM은 수 시간 안에 착수했고, CBM은 대체로 며칠 뒤에 착수**했다고 볼 수 있다.

### 해석(가설)
- CBM은 징후를 보고 작업을 **계획해서** 나중에 착수하기 때문일수도
    - 다만 기록된 감지 → 작업착수 시간과 모델이 미리 알려줄 수 있는 시간은 같은 값이 아님. 실제 예측 선행시간으로 쓰려면 센서 징후 시점을 따로 확인해야 함.
- 다만 반대 근거도 있음
    - CBM 중 REMARK가 **"설비 트립"인 2건도 약 68시간·110시간 뒤에 착수**  - 설비 트립(trip)은 설비가 보호장치에 걸려 자동으로 멈춘 것
    - "확인 중" OR "원인 미상" OR "재점검 필요" -> 원인을 찾는 중이었던 기록이 많다. 
        - 반대로 CBM 10건 중 2건은 하루 안에 착수했음 -> 그래서 "계획된 여유"라고 단정하지 말아야 함.

### 보고서에는 그래서 보고서에는 "계획된 여유일 수도 있다(가설)" 정도로만 쓰자는 의미가 될 수 있다 ㅇㅇ

### 2-3. 정비했다고 전부 멈춘 건가?

[정비 데이터사전](https://app.notion.com/p/26be609a45d0838fb0c88106c95f4c02)의 `DOWN_MIN`은 기록된 실제 정지시간(분)임. `END_DT - STRT_DT`는 작업시간이고 둘은 다름.

0분도 유효한 값임. 예비기 절체·무정지 점검 등이 가능하므로 정비유형만으로 정지를 판단하지 않겠음. 해당 작업의 0분 기록이 그 시간대 설비 전체의 운전을 증명하지도 않음. 실제 정지 시작·종료가 없어서 `STRT_DT + DOWN_MIN`이나 `END_DT - DOWN_MIN`으로 임의 구간을 만들지 않음.

In [ ]:
downtime_summary = maintenance_events.groupby('MNT_TYPE').agg(
    정비건수=('WO_NO', 'size'),
    정지양수=('DOWN_MIN', lambda s: int(s.gt(0).sum())),
    정지0분=('DOWN_MIN', lambda s: int(s.eq(0).sum())),
    정지결측=('DOWN_MIN', lambda s: int(s.isna().sum())),
    정지분중앙값=('DOWN_MIN', 'median'),
    작업분중앙값=('WORK_MIN', 'median'),
).reindex(['BM', 'TBM', 'CBM'])
for name, mask in {
    '작업시간과같음': maintenance_events.DOWN_MIN.eq(maintenance_events.WORK_MIN),
    '작업시간보다짧음': maintenance_events.DOWN_MIN.lt(maintenance_events.WORK_MIN),
    '작업시간보다김': maintenance_events.DOWN_MIN.gt(maintenance_events.WORK_MIN),
}.items():
    downtime_summary[name] = mask.groupby(maintenance_events.MNT_TYPE).sum().reindex(downtime_summary.index)
assert downtime_summary['정비건수'].sum() == len(maintenance_events)
assert downtime_summary[['정지양수', '정지0분', '정지결측']].sum(axis=1).eq(downtime_summary['정비건수']).all()
assert not maintenance_events.STOP_INTERVAL_KNOWN.any()
display(downtime_summary)
display(maintenance_events.loc[maintenance_events.DOWN_MIN.eq(0), [
    'WO_NO', 'MNT_TYPE', 'STRT_DT', 'END_DT', 'WORK_MIN', 'DOWN_MIN', 'REMARK', 'PART_CHG'
]])

온도 관측과 겹치는 **작업 구간**도 따로 확인함. 아래 행수는 `[착수, 완료)`에 기록된 센서 행수이며 정지 중 관측 행수가 아님. 겹치는 작업의 `DOWN_MIN`을 더해서 설비 전체 정지시간으로 쓰지 않음. 정기수리캘린더도 실제 정지 로그와 구분함.

In [ ]:
lot_context = build_lot_context(temp, event)
marked_temp, _ = apply_user_exclusions(temp)
overlap_records = []
for work in maintenance_events.itertuples(index=False):
    # 작업 시간과 실제 관측의 겹침만 계산함. 정지 시간을 이 위치로 배정하지 않음.
    hit = marked_temp.loc[marked_temp.MEAS_DT.ge(work.STRT_DT) & marked_temp.MEAS_DT.lt(work.END_DT)]
    for lot_no, observed in hit.groupby('LOT_NO'):
        overlap_records.append({
            'LOT_NO': lot_no, 'WO_NO': work.WO_NO, 'MNT_TYPE': work.MNT_TYPE,
            'STRT_DT': work.STRT_DT, 'END_DT': work.END_DT,
            'WORK_MIN': work.WORK_MIN, 'DOWN_MIN': work.DOWN_MIN,
            'DOWN_RECORD_STATUS': work.DOWN_RECORD_STATUS, 'STOP_INTERVAL_KNOWN': work.STOP_INTERVAL_KNOWN,
            'OBSERVED_START': observed.MEAS_DT.min(), 'OBSERVED_END': observed.MEAS_DT.max(),
            'OBSERVED_WORK_OVERLAP_ROWS': len(observed),
            'RETAINED_WORK_OVERLAP_ROWS': int((~observed.EXCLUDED).sum()),
            'PART_CHG': work.PART_CHG, 'PART_NAMES': work.PART_NAMES,
        })
work_observation_overlap = pd.DataFrame(overlap_records)
assert not work_observation_overlap.duplicated(['LOT_NO', 'WO_NO']).any()
display(work_observation_overlap)
downtime_summary.to_csv(OUTPUT_DIR / '02_downtime_summary.csv', encoding='utf-8-sig')
work_observation_overlap.to_csv(OUTPUT_DIR / '02_work_observation_overlap.csv', index=False, encoding='utf-8-sig')

### 2-4. 1년 동안 언제 정비했나?

**왜 이 그래프가 필요한가?**
- 정비 44건이 1년에 고르게 퍼져 있는지, **특정 시기에 몰려 있는지** 봐야함 
- 특히 CR1 정기수리(6월, 11월) 기간에 한 정비는 이상 징후 때문이 아니라 수리 일정에 따른 작업일 수도 있어서, 뒤에서 온도 이상 후보와 비교할 때 **섞이면 안 되는 정비**
- 이 그래프로 정기수리와 겹치는 정비를 먼저 확인함

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.2))

# 회색 = CR1 정기수리 (종료일 당일까지 포함하려고 +1일)
for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

# 정비유형별로 한 줄씩 점 찍기
for i, t in enumerate(MNT_ORDER):
    sub = cal[cal["MNT_TYPE"] == t]
    ax.scatter(sub["DETC_DT"], [i] * len(sub), color=MNT_COLORS[t], s=45, edgecolor="white", zorder=3, label=t)

ax.set_yticks([0, 1, 2], MNT_ORDER)
ax.set_ylim(-0.7, 2.7)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 정비 탐지 시점 (회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "02_mnt_timeline.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 정기수리 기간(종료일 포함) 안에 탐지된 정비
cal["정기수리중"] = False
for _, r in cr1_repair.iterrows():
    inside = (cal["DETC_DT"] >= r["STRT_DT"]) & (cal["DETC_DT"] < r["END_DT"] + pd.Timedelta(days=1))
    cal.loc[inside, "정기수리중"] = True

print("정기수리 기간 안에 탐지된 정비:", cal["정기수리중"].sum(), "건")
display(cal.loc[cal["정기수리중"], ["WO_NO", "DETC_DT", "MNT_TYPE", "FAIL_MODE", "REMARK"]])

**읽는 법:** 
- 회색(정기수리) 기간에 탐지된 11건은 모두 **TBM(파란색)**
- 그중 9건은 REMARK가 "수리기간 중 점검", "정기수리 부속작업", "SD작업 수행"처럼 **수리 일정에 따른 작업**
- 나머지 2건은 실제 결함 기록이 있음(`WO-2024-00931` 마모량 초과, `WO-2024-00916` O링 세트 손상 교체)
- 이상 징후 때문에 한 정비와 섞어서 해석하지 않도록 `정기수리중` 컬럼으로 표시해 두겠음. 라벨 근거를 검토할 때 참고함

> 여기 그림은 정기수리 종료일을 포함하는 가정으로 표시했음. 종료시각의 해석은 확인이 필요하고, 두 기준의 차이는 01에서 비교했음.

## 3. Step 2 · `WO_NO`로 부품교체이력 연결

- 부품교체이력에는 **날짜가 없음** 
- `WO_NO`(작업지시번호)로 정비이력에 붙여야 언제, 어떤 정비에서 교체했는지 알 수 있음

**왜 먼저 두 파일이 맞는지 확인하는가?** 
- 연결 키가 한쪽에만 있거나 `PART_CHG` 표시와 실제 교체기록이 어긋나면, 붙인 결과를 믿을 수 없음.
    - 연결 전에 전체 데이터에서 한 번 확인해야 함

In [ ]:
wo_y = set(maint.loc[maint["PART_CHG"] == "Y", "WO_NO"])  # 정비이력: 부품교체 = Y 인 작업지시
wo_chg = set(partchg["WO_NO"])                            # 부품교체이력에 등장한 작업지시

print("정비이력 PART_CHG=Y:", len(wo_y), "건 / 부품교체이력 WO_NO:", len(wo_chg), "개")
print("Y인데 교체기록 없음:", len(wo_y - wo_chg), "건")
print("교체기록 있는데 Y가 아님:", len(wo_chg - wo_y), "건")
print("작업지시 1건당 교체 부품 행 수 최대:", partchg["WO_NO"].value_counts().max())

In [ ]:
# 정비 표는 44건을 유지함. 한 WO에서 부품이 여러 개여도 정비 행을 늘리지 않음.
# 교체 상세는 별도 표로 확인함. 여기서는 8개 WO에 9개 부품 행이 연결됨.
cal_chg = part_details.copy()
print('교체 없는 정비까지 남긴 정비:', len(maintenance_events), '건')
print('부품교체가 붙은 정비:', cal_chg['WO_NO'].nunique(), '건 → 교체 행:', len(cal_chg), '행')
display(cal_chg[['WO_NO', 'EQP_TAG', 'DETC_DT', 'MNT_TYPE', 'FAIL_MODE', 'PART_CD', 'QTY', 'CHG_RSN', 'REMARK']])
assert maintenance_events.WO_NO.is_unique
assert len(maintenance_events) == len(maint.loc[maint.EQP_TAG.eq(CAL_TAG)])

**왜 정비유형 × 교체사유를 보는가?**
- 같은 부품 교체라도 "고장 나서" 바꾼 것과 "수명이 돼서/예방으로" 바꾼 것은 의미가 다르다.
- 손상 교체가 어떤 정비유형에서 나왔는지 알면, 어떤 교체 기록을 센서와 함께 검토할지 좁힐 수 있음. 해당 기록만으로 사전 탐지가 가능했다고 단정하지 않겠음.

In [ ]:
cross = pd.crosstab(cal_chg["MNT_TYPE"], cal_chg["CHG_RSN"]).reindex(MNT_ORDER)
display(cross)

**읽는 법** 
- 정비 8건에서 9행이 나온 이유는 `WO-2024-00916` 한 건에서 부품 2종을 교체했기 때문
- 손상 교체 4건 중 3건이 BM(고장정비)
- `QTY` = **한 작업지시에서 해당 부품을 교체한 수량(개)** (데이터 사전 정의)

## 4. Step 3 · `PART_CD`로 부품마스터 연결

부품 코드에 이름 | 분류 |적용설비 | 표준수명(`STD_LIFE_H`, 시간) | 조달기간(`LEAD_TIME_D`, 일)을 붙여보자

In [ ]:
cal_part = part_details.copy()  # 공통 함수에서 PART_CD로 부품마스터까지 연결했음
print("부품마스터에서 못 찾은 PART_CD:", cal_part["PART_NM"].isna().sum(), "건")

show_cols = ["WO_NO", "DETC_DT", "MNT_TYPE", "PART_CD", "PART_NM", "PART_TYPE",
             "APPLY_EQP_TYPE", "STD_LIFE_H", "LEAD_TIME_D", "QTY", "CHG_RSN"]
display(cal_part[show_cols])

In [ ]:
# 부품별 요약: 교체 횟수(count)와 총 수량(sum)
display(cal_part.groupby(["PART_CD", "PART_NM"])["QTY"].agg(["count", "sum"]))

### 4-1. 어떤 부품을 언제 교체했나?

**왜 이 그래프가 필요한가?**
- 표로는 9행이 흩어져 보이지만, 부품별로 한 줄씩 날짜 위에 찍으면 **같은 부품이 몇 번, 얼마 간격으로 반복 교체되는지**가 바로 보임
- 또 교체가 정기수리 기간에 몰렸는지, 고장정비(BM)에서 나왔는지도 함께 확인할 수 있음

그래프 보는 법: 점 색 = 정비유형(TBM, BM, CBM), 옆 글자 = `교체사유 ×수량`.

부품교체이력에는 실제 교체 시각이 없음. 아래 점은 연결된 정비 작업의 종료시각으로 표시한 것임.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

for _, r in cr1_repair.iterrows():
    ax.axvspan(r["STRT_DT"], r["END_DT"] + pd.Timedelta(days=1), color="#d9d9d9", lw=0)

for t in MNT_ORDER:
    sub = cal_part[cal_part["MNT_TYPE"] == t]
    ax.scatter(sub["END_DT"], sub["PART_NM"], color=MNT_COLORS[t], s=90, edgecolor="white", zorder=3, label=t)

for _, r in cal_part.iterrows():
    ax.annotate(f"{r['CHG_RSN']} ×{r['QTY']}", (r["END_DT"], r["PART_NM"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8)

ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-25"))  # 12월 글자가 잘리지 않게 오른쪽 여유
ax.set_ylim(-0.6, 2.6)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_title("연속소둔로 부품 교체 시점 (작업완료 END_DT 기준, 회색 = CR1 정기수리)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.savefig(os.path.join(FIG_DIR, "02_part_change_timeline.png"), bbox_inches="tight")
plt.show()

**읽는 법**
- 교체된 부품은 **3종뿐**이고, 3종 모두 1년에 **3번씩** 교체되었음
- 6월 10일(정기수리 첫날)에는 오일씰과 O링 세트를 같이 바꿨음

### 4-2. 같은 부품 코드의 기록 간격과 표준수명 비교

부품마스터의 표준수명(`STD_LIFE_H`)과 같은 부품 코드의 기록 간격을 참고로 비교해보자.

계산: 같은 `PART_CD`의 직전 정비 완료(`END_DT`) → 이번 정비 완료.
- 부품마다 첫 기록은 그 전 기록이 없어서 계산에서 빼겠음.
- 어느 위치의 부품인지, 같은 개체를 다시 바꾼 것인지 알 수 없음.
- 그래서 이 간격을 실제 부품 수명이나 잔여수명으로 부르면 안 됨. 조기 마모 여부도 이 값만으로 확정하지 않겠음.

In [ ]:
cal_part = cal_part.sort_values("END_DT")
cal_part["동일코드_기록간격_h"] = cal_part.groupby("PART_CD")["END_DT"].diff().dt.total_seconds() / 3600
cal_part["표준수명대비_기록간격_%"] = cal_part["동일코드_기록간격_h"] / cal_part["STD_LIFE_H"] * 100

gaps = cal_part.dropna(subset=["동일코드_기록간격_h"])
display(gaps[["PART_CD", "PART_NM", "END_DT", "CHG_RSN", "동일코드_기록간격_h", "STD_LIFE_H", "표준수명대비_기록간격_%"]].round({"동일코드_기록간격_h": 1, "표준수명대비_기록간격_%": 1}))

In [ ]:
life = cal_part.groupby("PART_NM")["STD_LIFE_H"].first()  # 부품별 표준수명

fig, ax = plt.subplots(figsize=(11, 3))
ax.barh(life.index, life.values, color="#d9d9d9", height=0.5, label="표준수명 (STD_LIFE_H)")
ax.scatter(gaps["동일코드_기록간격_h"], gaps["PART_NM"], color="#2a78d6", s=70, zorder=3, label="동일 부품코드 기록 간격")
ax.set_xlabel("시간 (h)")
ax.set_title("표준수명(회색 막대) vs 동일 부품코드 기록 간격(파란 점)")
ax.legend(loc="lower right")
plt.savefig(os.path.join(FIG_DIR, "02_life_vs_interval.png"), bbox_inches="tight")
plt.show()

**읽는 법** 
- 세 부품 코드 모두 **기록 간격 / 표준수명 값이 40% 미만**이었음. 실제 같은 개체의 사용시간은 모르므로 수명 소진율로 읽지 않겠음.

그래도 "빨리 닳았다"고 단정할 수 없는 이유가 있다
- 데이터 사전: `STD_LIFE_H`는 설계 표준값(설정값)이라 실제 교체 주기와 원래 일치하지 않음
- 재교체 6번 중 **손상 교체는 3번뿐**이고, 나머지는 예방 2번, 수명도래 1번
    - 예방 교체는 닳아서 바꾼 것이 아님
- 동일 부품 코드의 기록 간격 / 표준수명 값이 약 23%인데 사유가 "수명도래"인 기록(오일씰, 6/10)이 있어, `CHG_RSN`과 `STD_LIFE_H`의 수명 기준이 서로 다를 수 있음
- `QTY`가 1~3개로 다르고 **어느 위치의 부품인지** 기록이 없습니다.

### 그래서 같은 부품 코드의 기록 간격으로만 두자

동일한 위치·개체의 사용수명을 확인한 값은 아님.

### 4-3. 연속로 전용 부품 중 실제로 교체된 것은?

**왜 이 그래프가 필요한가?**
- 부품마스터에는 "공통" 57개, "연속로" 11개가 있다 
- 이 68개를 전부 분석할 필요가 있는지, 그리고 **온도 채널과 직접 관련된 부품(히터, 열전대 등)** 이 실제로 교체된 적이 있는지 확인해야 함 
- 온도 이상 후보를 나중에 "부품 교체 기록"으로 검증할 수 있는지가 여기서 결정됨

그래프 보는 법
- 연속로 전용 부품 11개
    - 파랑 = 2024년 CAL01에서 교체됨
    - 회색 = 교체 기록 없음.

In [ ]:
furnace = part[part["APPLY_EQP_TYPE"] == "연속로"].copy()
furnace["교체됨"] = furnace["PART_CD"].isin(cal_part["PART_CD"])
furnace = furnace.sort_values("STD_LIFE_H")

labels = furnace["PART_CD"] + " " + furnace["PART_NM"]
colors = ["#2a78d6" if x else "#c3c2b7" for x in furnace["교체됨"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
axes[0].barh(labels, furnace["STD_LIFE_H"], color=colors)
axes[0].set_title("표준수명 STD_LIFE_H (시간)")
axes[1].barh(labels, furnace["LEAD_TIME_D"], color=colors)
axes[1].set_title("조달기간 LEAD_TIME_D (일)")
fig.suptitle("연속로 전용 부품 11개 (파랑 = 2024년 CAL01에서 교체, 회색 = 교체 기록 없음)", y=1.02)
plt.savefig(os.path.join(FIG_DIR, "02_furnace_parts.png"), bbox_inches="tight")
plt.show()

print("공통 + 연속로 부품 수:", part["APPLY_EQP_TYPE"].isin(["공통", "연속로"]).sum(), "개")
print("그중 CAL01에서 실제 교체된 부품:", sorted(cal_part["PART_CD"].unique()))

**읽는 법 (도메인 관점)**

- 실제 교체된 부품은 **오일씰(SEL-006, 공통), O링 세트(SEL-009, 공통), 가스 분석계 셀(INS-006, 연속로)** 3종뿐
    - 공통/연속로 후보 68개 중 **교체 기록으로 검증할 수 있는 것은 이 3종뿐**
    - 오일씰과 O링은 공장 전체에서 29번, 36번 교체된 흔한 공통 실링 부품
    -  연속소둔로에서 이 부품을 바꾼 BM은 고장모드가 ELE(인버터 알람·모터 과열·절연 저하) 
        - 온도 채널(TC·OP·CP)과 직접 이어지는 근거는 없다
---
- 온도 채널과 가장 직접 관련된 **전기히터 소자 Z1~Z3, 열전대, CP 산소 프로브, 라디언트 튜브**는 2024년 교체 기록이 없다
    - 그래서 TC·OP 채널에서 이상 후보가 나와도 "히터/열전대를 교체했다"는 기록으로 검증할 사례는 없다
    - 하지만! 교체 기록이 없다고 **열화가 없었다는 뜻은 아니며**, 교체 없이 서서히 나빠지는 것을 찾는 것이 바로 이상탐지의 역할이 된다

---

- `CP-Z4`는 데이터 사전상 **카본포텐셜** (가열·균열존 분위기 제어값). 단위와 구체적 의미는 확인 필요 (`[확인필요]`) 
    - 가스 분석계 셀(INS-006)도 분위기 계측 부품이라 `CP-Z4`와 연결해 볼 후보 (사전에 둘의 관계는 없음 → 추정)

- 교체된 3종 중 조달기간이 가장 긴 가스 분석계 셀(31일)은, 예비품·조달 계획과 함께 검토할 수 있음. 조달기간을 모델의 필수 예측 선행시간으로 바로 정하지는 않겠음
    - 연속로 부품 중에는 퍼니스롤(85일), 라디언트 튜브(72일)처럼 더 긴 부품도 있습니다. CBM 경보 기준을 정할 때 쓸 수 있는 값임

## 5. 다음 분석에 넘길 정비 표

정비는 1건당 1행, 부품은 교체 품목당 1행으로 따로 저장함.
부품 상세를 온도 전체에 바로 붙여서 센서 행을 복제하지 않겠음.

정비 결과와 부품교체 내용은 사후 검토 근거임. 센서의 과거 시점에 이미 알고 있던 입력이라고 취급하면 안 됨.
다음은 [03 LOT별 센서 비교](03_lot_comparison.ipynb)이고, 라벨 근거는 [05 구간 검토](05_interval_review.ipynb)에 모을거임.

In [ ]:
maintenance_events.to_csv(OUTPUT_DIR / '02_maintenance_events.csv', index=False, encoding='utf-8-sig')
part_details.to_csv(OUTPUT_DIR / '02_part_details.csv', index=False, encoding='utf-8-sig')
print('정비', len(maintenance_events), '건 / 교체 없는 정비', (~maintenance_events.HAS_PART_RECORD).sum(), '건 / 부품 상세', len(part_details), '행')